# EMEA AI Cohort — Assignment 7
## Random Forest Feature Selection

**Objective:** Use a Random Forest model to estimate feature importance, select a smaller set of useful features, and compare model performance using all features versus selected features.

### Workflow
1. Load a public classification dataset.
2. Inspect and prepare the data.
3. Split the data into training and test sets.
4. Train a baseline Random Forest.
5. rank features using Random Forest feature importance.
6. Select a subset of features.
7. Compare held-out test-set metrics for both models.
8. Plot feature importance and discuss the results.

**Dataset choice:** The scikit-learn Breast Cancer Wisconsin Diagnostic dataset is bundled with scikit-learn, so no separate CSV download is needed. It has 30 numeric features and a binary target. The dataset is for a machine-learning exercise, not for medical diagnosis.

**Results policy:** The notebook calculates metrics when you run it. No accuracy or feature rankings are hard-coded as experimental results.

## 1. Import libraries

We use `RandomForestClassifier` for classification and feature importance, `train_test_split` for a held-out test set, and scikit-learn metrics for evaluation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
print("Libraries imported successfully.")

## 2. Load and inspect the dataset

The dataset is included with scikit-learn. Its target labels are `0 = malignant` and `1 = benign`. We keep the original target encoding and explicitly print the mapping to avoid confusion.

In [ ]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data.copy()
y = dataset.target.copy()

target_names = {0: "malignant", 1: "benign"}

print("Dataset shape:", X.shape)
print("Number of features:", X.shape[1])
print("Target mapping:", target_names)
print("\nClass counts:")
display(y.map(target_names).value_counts().to_frame("count"))
print("\nFirst five rows:")
display(X.head())

print("\nMissing values:", int(X.isna().sum().sum()))

In [ ]:
# Visualize target distribution.
class_counts = y.map(target_names).value_counts()
plt.figure(figsize=(6, 4))
plt.bar(class_counts.index, class_counts.values)
plt.title("Target Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of samples")
plt.tight_layout()
plt.show()

## 3. Create training and test sets

We use an 80/20 split. `stratify=y` keeps the class proportions similar across the training and test sets. Feature selection will be performed using only the training set to reduce data leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))
print("Training features:", X_train.shape[1])

## 4. Train the baseline Random Forest

A Random Forest fits many decision trees on randomized samples/features and combines their predictions. We train the baseline model using all 30 features. The `n_jobs=-1` option uses available CPU cores.

In [ ]:
baseline_model = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1
)
baseline_model.fit(X_train, y_train)

print("Baseline Random Forest trained using all features.")

## 5. Rank features by Random Forest importance

`feature_importances_` measures the average reduction in impurity contributed by each feature across the fitted trees. It is useful as a first-pass ranking, but it can favor some kinds of features and should not be interpreted as causal importance.

In [ ]:
importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": baseline_model.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("Feature importance ranking:")
display(importance_df)

TOP_K = 10  # You can change this value to select more or fewer features.
selected_features = importance_df.head(TOP_K)["feature"].tolist()

print(f"\nSelected {len(selected_features)} features:")
for rank, feature in enumerate(selected_features, start=1):
    print(f"{rank}. {feature}")

In [ ]:
# Plot the top features.
top_plot = importance_df.head(TOP_K).sort_values("importance", ascending=True)
plt.figure(figsize=(9, 6))
plt.barh(top_plot["feature"], top_plot["importance"])
plt.xlabel("Random Forest feature importance")
plt.title(f"Top {TOP_K} Features")
plt.tight_layout()
plt.show()

## 6. Train a Random Forest using selected features

We train a second model using only the selected features. Both models use the same train/test split and the same Random Forest settings so the comparison is easier to interpret.

In [ ]:
selected_model = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1
)
selected_model.fit(X_train[selected_features], y_train)

print("Selected-feature Random Forest trained.")

## 7. Evaluate and compare models

The metrics below are computed from the held-out test set. ROC-AUC uses predicted probabilities rather than hard class labels. Results can vary if you change the split, dataset, or model settings.

In [ ]:
def evaluate_model(model, X_eval, y_eval, model_name):
    predictions = model.predict(X_eval)
    probabilities = model.predict_proba(X_eval)[:, 1]
    return {
        "Model": model_name,
        "Features used": X_eval.shape[1],
        "Accuracy": accuracy_score(y_eval, predictions),
        "Precision (benign class)": precision_score(y_eval, predictions, pos_label=1, zero_division=0),
        "Recall (benign class)": recall_score(y_eval, predictions, pos_label=1, zero_division=0),
        "F1-score (benign class)": f1_score(y_eval, predictions, pos_label=1, zero_division=0),
        "ROC-AUC": roc_auc_score(y_eval, probabilities)
    }

comparison = pd.DataFrame([
    evaluate_model(baseline_model, X_test, y_test, "All features"),
    evaluate_model(selected_model, X_test[selected_features], y_test, f"Top {TOP_K} features")
])

display(comparison.round(4))

In [ ]:
# Confusion matrices for both models.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_estimator(
    baseline_model, X_test, y_test,
    display_labels=["malignant", "benign"],
    ax=axes[0], values_format="d"
)
axes[0].set_title("All Features")

ConfusionMatrixDisplay.from_estimator(
    selected_model, X_test[selected_features], y_test,
    display_labels=["malignant", "benign"],
    ax=axes[1], values_format="d"
)
axes[1].set_title(f"Top {TOP_K} Features")
plt.tight_layout()
plt.show()

## 8. Generate a results summary from this run

This cell prints the actual metric values and a neutral comparison. It does not assume feature selection improves performance: fewer features may perform similarly, better, or worse.

In [ ]:
print("RESULTS SUMMARY — values calculated from this run")
print("=" * 60)
for _, row in comparison.iterrows():
    print(f"\n{row['Model']}")
    print(f"  Features used: {int(row['Features used'])}")
    for metric in ["Accuracy", "Precision (benign class)", "Recall (benign class)", "F1-score (benign class)", "ROC-AUC"]:
        print(f"  {metric}: {row[metric]:.4f}")

all_accuracy = comparison.loc[comparison["Model"] == "All features", "Accuracy"].iloc[0]
selected_accuracy = comparison.loc[comparison["Model"] == f"Top {TOP_K} features", "Accuracy"].iloc[0]
delta = selected_accuracy - all_accuracy

print("\nAccuracy difference (selected minus all):", f"{delta:+.4f}")
if delta > 0:
    print("On this split, the selected-feature model had higher accuracy.")
elif delta < 0:
    print("On this split, the all-feature model had higher accuracy.")
else:
    print("Both models had the same accuracy on this split.")

print("\nThis is a single held-out split, not proof of general superiority.")

## 9. Conclusion template

Complete this paragraph after running the notebook, using the values printed in your output:

> A Random Forest classifier was trained on the Breast Cancer Wisconsin Diagnostic dataset. The baseline model used **30 features**, while the feature-selection model used **[TOP_K] features** selected from the training data by Random Forest impurity-based importance. On the held-out test set, the all-feature model achieved an accuracy of **[copy output]**, and the selected-feature model achieved an accuracy of **[copy output]**. The accuracy difference was **[copy output]**. Therefore, for this split, feature selection **[improved / matched / reduced]** accuracy while using fewer inputs. These findings are limited to this dataset and split; the model is not suitable for medical decision-making.

## 10. Key concepts

- **Feature selection:** reducing the number of input variables used by a model.
- **Random Forest feature importance:** a tree-based ranking based on impurity reduction.
- **Data leakage prevention:** rank features using training data only.
- **Held-out evaluation:** compare both models on the same unseen test examples.
- **Limitations:** correlated features can share importance, impurity importance can be biased, and a single split does not prove robust performance.

### References
- scikit-learn Breast Cancer Wisconsin dataset: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html
- RandomForestClassifier documentation: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
- UCI Breast Cancer Wisconsin (Diagnostic) dataset: https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic